# 3. Анализ результатов\n\nФинальные метрики, confusion matrix, примеры детекций (TP/FP/FN).

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from ultralytics import YOLO

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

PROJECT_ROOT = Path('..').resolve()
EXP_NAME = 'fire_v1'
WEIGHTS = PROJECT_ROOT / 'runs' / 'detect' / EXP_NAME / 'weights' / 'best.pt'
CLASS_NAMES = {0: 'fire', 1: 'smoke', 2: 'false_source'}

## 3.1 Валидация на тестовом split

In [ ]:
if WEIGHTS.exists():
    model = YOLO(str(WEIGHTS))
    
    results = model.val(
        data=str(PROJECT_ROOT / 'configs' / 'dataset.yaml'),
        split='test',
        imgsz=1024,
        plots=True,
        save_json=True,
    )
    
    print(f"mAP@0.5:      {results.box.map50:.4f}")
    print(f"mAP@0.5:0.95: {results.box.map:.4f}")
    
    # Таблица по классам
    print(f"\n{'Класс':<15} {'P':>8} {'R':>8} {'F1':>8} {'mAP50':>8} {'mAP50-95':>8}")
    print("-" * 63)
    for i, name in CLASS_NAMES.items():
        if i < len(results.box.ap50):
            p = results.box.p[i] if i < len(results.box.p) else 0
            r = results.box.r[i] if i < len(results.box.r) else 0
            f1 = 2 * p * r / (p + r + 1e-16)
            print(f"{name:<15} {p:>8.4f} {r:>8.4f} {f1:>8.4f} {results.box.ap50[i]:>8.4f} {results.box.ap[i]:>8.4f}")
else:
    print(f"Веса не найдены: {WEIGHTS}")
    print("Сначала обучите модель: python src/train.py")

## 3.2 Confusion Matrix

In [ ]:
# Confusion matrix из результатов ultralytics
cm_path = WEIGHTS.parent.parent / 'confusion_matrix_normalized.png'
if not cm_path.exists():
    # Поиск в последней валидации
    val_dirs = sorted((PROJECT_ROOT / 'runs' / 'detect').glob('val*'))
    for vd in reversed(val_dirs):
        cp = vd / 'confusion_matrix_normalized.png'
        if cp.exists():
            cm_path = cp
            break

if cm_path.exists():
    img = cv2.imread(str(cm_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title('Нормализованная Confusion Matrix', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Confusion matrix не найдена. Будет создана после запуска валидации.")

## 3.3 Примеры детекций на тестовых изображениях

In [ ]:
import random

if WEIGHTS.exists():
    model = YOLO(str(WEIGHTS))
    test_img_dir = PROJECT_ROOT / 'data' / 'processed' / 'test' / 'images'
    test_images = list(test_img_dir.glob('*.png')) + list(test_img_dir.glob('*.jpg'))
    
    if test_images:
        random.seed(42)
        samples = random.sample(test_images, min(8, len(test_images)))
        
        fig, axes = plt.subplots(2, 4, figsize=(20, 10))
        axes = axes.flatten()
        
        for idx, img_path in enumerate(samples):
            results = model.predict(str(img_path), imgsz=1024, conf=0.25, verbose=False)
            annotated = results[0].plot()
            annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
            
            axes[idx].imshow(annotated_rgb)
            axes[idx].set_title(img_path.name, fontsize=9)
            axes[idx].axis('off')
        
        for idx in range(len(samples), len(axes)):
            axes[idx].axis('off')
        
        plt.suptitle('Примеры детекций на тестовых изображениях', fontsize=14)
        plt.tight_layout()
        plt.savefig(str(PROJECT_ROOT / 'runs' / 'test_predictions.png'), dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print("Нет тестовых изображений")
else:
    print("Модель не обучена")